In [3]:
# Cell 1 — Setup & inputs (GFM Flood STAC pipeline)
# User inputs: ISO3 + reference date (as-of)
# Notes:
# - Admin boundaries source of truth: admin2 layer in global_admin_boundaries_matched_latest.gdb.zip
# - WorldPop raster is provided manually per country (hardcoded path per ISO3)
# - Flood data comes from EODC STAC (GloFAS Global Flood Monitor / GFM)

from __future__ import annotations

from pathlib import Path
import os
import json
import datetime as dt

import numpy as np
import pandas as pd
import geopandas as gpd

# -----------------------
# USER INPUTS
# -----------------------
ISO3 = "SDN"  # <- set per run
AS_OF_DATE = "2025-11-30"  # <- YYYY-MM-DD (inclusive end date)
LOOKBACK_MONTHS = 12  # flood window length (months)

# -----------------------
# INPUT DATA PATHS
# -----------------------
# Admin boundaries (authoritative)
GDB_ZIP_PATH = Path("../COD/global_admin_boundaries_matched_latest.gdb.zip")
ADMIN2_LAYER = "admin2"

# WorldPop raster (hardcode per ISO3; update as needed)
WORLDPOP_PATH = Path("./population/sdn_pop_2025_CN_100m_R2025A_v1.tif")

# -----------------------
# STAC SETTINGS (GFM)
# -----------------------
STAC_API_URL = "https://stac.eodc.eu/api/v1"
GFM_COLLECTION_ID = "GFM"

# Asset / band name to load (we’ll confirm in later cell via item asset keys)
GFM_ASSET_KEY = "ensemble_flood_extent"  # expected name used in your current notebooks

# -----------------------
# OUTPUT BASE
# -----------------------
# Keep consistent with your other hazards: ../outputs/<pipeline>/<ISO3>/<run_id>/
OUTPUTS_ROOT = Path("../outputs")
PIPELINE_NAME = "gfm_flood_exposure"


# -----------------------
# BASIC VALIDATION
# -----------------------
def _assert_exists(p: Path, label: str) -> None:
    if not p.exists():
        raise FileNotFoundError(f"Missing {label}: {p.resolve()}")


_assert_exists(GDB_ZIP_PATH, "admin boundaries GDB zip")
_assert_exists(WORLDPOP_PATH, "WorldPop raster")

# Parse dates
AS_OF_DT = pd.to_datetime(AS_OF_DATE)
if pd.isna(AS_OF_DT):
    raise ValueError(f"AS_OF_DATE must be YYYY-MM-DD, got: {AS_OF_DATE}")

print("Inputs:")
print("  ISO3:", ISO3)
print("  AS_OF_DATE:", AS_OF_DATE)
print("  LOOKBACK_MONTHS:", LOOKBACK_MONTHS)
print("Paths:")
print("  GDB_ZIP_PATH:", GDB_ZIP_PATH.resolve())
print("  WORLDPOP_PATH:", WORLDPOP_PATH.resolve())
print("STAC:")
print("  STAC_API_URL:", STAC_API_URL)
print("  GFM_COLLECTION_ID:", GFM_COLLECTION_ID)
print("  GFM_ASSET_KEY:", GFM_ASSET_KEY)
print("Outputs:")
print("  OUTPUTS_ROOT:", OUTPUTS_ROOT.resolve())
print("  PIPELINE_NAME:", PIPELINE_NAME)

Inputs:
  ISO3: SDN
  AS_OF_DATE: 2025-11-30
  LOOKBACK_MONTHS: 12
Paths:
  GDB_ZIP_PATH: /Users/jamesebrown/Library/Mobile Documents/com~apple~CloudDocs/Code/WIA/COD/global_admin_boundaries_matched_latest.gdb.zip
  WORLDPOP_PATH: /Users/jamesebrown/Library/Mobile Documents/com~apple~CloudDocs/Code/WIA/GFM/population/sdn_pop_2025_CN_100m_R2025A_v1.tif
STAC:
  STAC_API_URL: https://stac.eodc.eu/api/v1
  GFM_COLLECTION_ID: GFM
  GFM_ASSET_KEY: ensemble_flood_extent
Outputs:
  OUTPUTS_ROOT: /Users/jamesebrown/Library/Mobile Documents/com~apple~CloudDocs/Code/WIA/outputs
  PIPELINE_NAME: gfm_flood_exposure


In [4]:
# Cell 2 — Directory scaffold + run metadata helpers (GFM Flood)

from pathlib import Path
import json
import hashlib

# -----------------------
# Run ID + base directory
# -----------------------
# Keep naming consistent with other hazards:
#   <ISO3>_<start>_<end>_m<months>
WINDOW_END = AS_OF_DT
WINDOW_START = (AS_OF_DT - pd.DateOffset(months=LOOKBACK_MONTHS)) + pd.Timedelta(days=1)

RUN_ID = f"{ISO3}_{WINDOW_START.date()}_{WINDOW_END.date()}_m{LOOKBACK_MONTHS}"
BASE_DIR = OUTPUTS_ROOT / PIPELINE_NAME / ISO3 / RUN_ID

# -----------------------
# Output directories
# -----------------------
DIRS = {
    "base": BASE_DIR,
    "raw": BASE_DIR / "raw",  # STAC manifests, item JSON, etc.
    "flood_native": BASE_DIR / "flood_native",  # native-grid flood masks / rasters
    "flood_worldpop": BASE_DIR
    / "flood_worldpop",  # flood masks reprojected to WorldPop grid
    "pop_exposed": BASE_DIR / "pop_exposed",  # population affected rasters
    "tables": BASE_DIR / "tables",  # admin2 CSVs
    "qc": BASE_DIR / "qc",  # figures + QC pdf
    "logs": BASE_DIR / "logs",  # manifests, debug logs
    "cache": OUTPUTS_ROOT
    / "_cache"
    / PIPELINE_NAME
    / ISO3,  # reusable per-ISO3 cache if needed
}


def ensure_dir(p: Path) -> Path:
    p.mkdir(parents=True, exist_ok=True)
    return p


for k, p in DIRS.items():
    ensure_dir(p)

# -----------------------
# Run metadata (JSON)
# -----------------------
RUN_METADATA_PATH = DIRS["base"] / "run_metadata.json"


def sha256_text(s: str) -> str:
    return hashlib.sha256(s.encode("utf-8")).hexdigest()


def update_run_metadata_artifact(key: str, path: Path, note: str | None = None) -> None:
    """
    Records an output artefact in run_metadata.json.
    Expects run_metadata dict to exist in scope.
    """
    run_metadata.setdefault("artefacts", {})
    run_metadata["artefacts"][key] = {
        "path": str(path),
        "note": note,
    }


# Initialise or load
if RUN_METADATA_PATH.exists():
    run_metadata = json.loads(RUN_METADATA_PATH.read_text())
else:
    run_metadata = {}

run_metadata.update(
    {
        "pipeline": PIPELINE_NAME,
        "iso3": ISO3,
        "as_of_date": AS_OF_DATE,
        "lookback_months": LOOKBACK_MONTHS,
        "window_start": str(WINDOW_START.date()),
        "window_end": str(WINDOW_END.date()),
        "stac_api_url": STAC_API_URL,
        "collection_id": GFM_COLLECTION_ID,
        "asset_key": GFM_ASSET_KEY,
        "worldpop_path": str(WORLDPOP_PATH),
        "admin_boundaries": {"path": str(GDB_ZIP_PATH), "layer": ADMIN2_LAYER},
    }
)

RUN_METADATA_PATH.write_text(json.dumps(run_metadata, indent=2))

print("Run:")
print("  RUN_ID:", RUN_ID)
print("  BASE_DIR:", BASE_DIR.resolve())
print("DIRS keys:", DIRS.keys())
print("Wrote:", RUN_METADATA_PATH)

Run:
  RUN_ID: SDN_2024-12-01_2025-11-30_m12
  BASE_DIR: /Users/jamesebrown/Library/Mobile Documents/com~apple~CloudDocs/Code/WIA/outputs/gfm_flood_exposure/SDN/SDN_2024-12-01_2025-11-30_m12
DIRS keys: dict_keys(['base', 'raw', 'flood_native', 'flood_worldpop', 'pop_exposed', 'tables', 'qc', 'logs', 'cache'])
Wrote: ../outputs/gfm_flood_exposure/SDN/SDN_2024-12-01_2025-11-30_m12/run_metadata.json


In [5]:
# Cell 3 — Admin2 boundaries (authoritative) + country AOI + bbox-hash
# Output:
#   - admin2_gdf: GeoDataFrame (filtered to ISO3)
#   - country_geom_4326: shapely geometry (union of admin2)
#   - stac_bbox: [W, S, E, N] (EPSG:4326)
#   - stac_intersects_geom: GeoJSON-like geometry for STAC intersects
#   - bounds_hash: stable hash for caching (ISO3 + bounds)
#   - bounds_hash_path: file written under logs/

import geopandas as gpd
from shapely.geometry import mapping, box

# ---- Load admin2 and filter ----
gdf = gpd.read_file(GDB_ZIP_PATH, layer=ADMIN2_LAYER)

if "iso3" not in gdf.columns:
    raise KeyError(
        f"'iso3' column not found in admin2 layer. Columns: {list(gdf.columns)}"
    )

admin2_gdf = gdf[gdf["iso3"] == ISO3].copy()
if admin2_gdf.empty:
    raise ValueError(f"No admin2 features found for ISO3={ISO3}")

# Ensure pcode column exists
if "adm2_pcode" not in admin2_gdf.columns:
    raise KeyError(
        f"'adm2_pcode' not found in admin2 layer. Columns: {list(admin2_gdf.columns)}"
    )

# Ensure CRS is set; reproject to EPSG:4326 for bbox/intersects
if admin2_gdf.crs is None:
    raise ValueError("Admin2 layer CRS is missing; cannot proceed.")

admin2_4326 = admin2_gdf.to_crs("EPSG:4326")

# ---- Union geometry (authoritative AOI) ----
# Avoid unary_union deprecation warnings by using union_all() when available
if hasattr(admin2_4326.geometry, "union_all"):
    country_geom_4326 = admin2_4326.geometry.union_all()
else:
    country_geom_4326 = admin2_4326.unary_union  # fallback for older geopandas/shapely

# ---- Bounds + STAC bbox/intersects ----
minx, miny, maxx, maxy = country_geom_4326.bounds
stac_bbox = [minx, miny, maxx, maxy]  # W, S, E, N
stac_intersects_geom = mapping(country_geom_4326)

print("Country bounds (EPSG:4326):")
print(f"  West={minx:.6f}, South={miny:.6f}, East={maxx:.6f}, North={maxy:.6f}")
print("STAC bbox:", stac_bbox)

# ---- Bounds hash for caching ----
bounds_hash_payload = {
    "iso3": ISO3,
    "layer": ADMIN2_LAYER,
    "bounds": {"west": minx, "south": miny, "east": maxx, "north": maxy},
}
bounds_hash = sha256_text(json.dumps(bounds_hash_payload, sort_keys=True))

bounds_hash_path = DIRS["logs"] / f"{ISO3}_bounds_hash.txt"
bounds_hash_path.write_text(bounds_hash)

update_run_metadata_artifact(
    "bounds_hash", bounds_hash_path, "Hash of admin2-union bounds for caching"
)
run_metadata["bounds_hash"] = bounds_hash
run_metadata["aoi_bounds_epsg4326"] = {
    "west": minx,
    "south": miny,
    "east": maxx,
    "north": maxy,
}
RUN_METADATA_PATH.write_text(json.dumps(run_metadata, indent=2))

print("bounds_hash:", bounds_hash)
print("Wrote:", bounds_hash_path)

/Users/jamesebrown/miniconda3/envs/gfm-stac/lib/python3.11/site-packages/pyogrio/raw.py:198: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts.  The processing may be really slow.  You can skip the processing by setting METHOD=SKIP.
  return ogr_read(


Country bounds (EPSG:4326):
  West=21.814556, South=8.637559, East=38.580036, North=23.146886
STAC bbox: [21.81455583500002, 8.637558893000005, 38.58003616299999, 23.146886230000007]
bounds_hash: a16a260f540850653cdbe4973e4a97d4e8866d26d541ba4edfe4f14ba4ab6307
Wrote: ../outputs/gfm_flood_exposure/SDN/SDN_2024-12-01_2025-11-30_m12/logs/SDN_bounds_hash.txt


In [6]:
# Cell 4 — STAC client setup (hard-coded GFM collection) + JSON-safe metadata

from pystac_client import Client

STAC_API_URL = "https://stac.eodc.eu/api/v1"
GFM_COLLECTION_ID = "GFM"  # hard-coded


def _json_safe(obj):
    """
    Convert common non-JSON types (datetime, pandas Timestamp, numpy types) to JSON-safe.
    """
    import datetime as _dt
    import numpy as _np
    import pandas as _pd

    if obj is None:
        return None

    # datetimes
    if isinstance(obj, (_dt.datetime, _dt.date)):
        return obj.isoformat()
    if isinstance(obj, _pd.Timestamp):
        return obj.isoformat()

    # numpy scalars
    if isinstance(obj, (_np.integer,)):
        return int(obj)
    if isinstance(obj, (_np.floating,)):
        return float(obj)

    # mappings / iterables
    if isinstance(obj, dict):
        return {str(k): _json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_json_safe(v) for v in obj]

    # fallback: stringify unknown objects rather than failing
    return obj


# ---- Connect to STAC ----
stac_client = Client.open(STAC_API_URL)
print("Connected to STAC:", STAC_API_URL)

# ---- Fetch collection explicitly ----
try:
    collection = stac_client.get_collection(GFM_COLLECTION_ID)
except Exception as e:
    raise RuntimeError(
        f"Failed to load GFM collection '{GFM_COLLECTION_ID}'. "
        "This likely indicates a breaking change in the STAC catalogue."
    ) from e

print("Loaded GFM collection:")
print("  ID:   ", collection.id)
print("  Title:", collection.title)

# ---- JSON-safe metadata capture ----
run_metadata["gfm_collection"] = _json_safe(
    {
        "id": collection.id,
        "title": collection.title,
        "description": collection.description,
        "license": collection.license,
        "spatial_extent": (
            collection.extent.spatial.bboxes if collection.extent else None
        ),
        "temporal_extent": (
            collection.extent.temporal.intervals if collection.extent else None
        ),
    }
)

RUN_METADATA_PATH.write_text(json.dumps(run_metadata, indent=2))

Connected to STAC: https://stac.eodc.eu/api/v1
Loaded GFM collection:
  ID:    GFM
  Title: Global Flood Monitoring


6188

In [7]:
# Cell 5 — STAC item search (12-month window) + manifest (idempotent)
# Output:
#   - items: list[pystac.Item]
#   - items_df: DataFrame with id, datetime, collection, asset keys
#   - manifest JSON/CSV saved under DIRS["logs"]

from shapely.geometry import shape
from datetime import timezone

# ---- Search window (inclusive end) ----
# Use daily time range (STAC expects RFC3339)
start_dt = (
    AS_OF_DT - pd.DateOffset(months=LOOKBACK_MONTHS) + pd.Timedelta(days=1)
).to_pydatetime()
end_dt = AS_OF_DT.to_pydatetime()

# Normalise to UTC (safe for STAC comparisons)
start_dt = start_dt.replace(tzinfo=timezone.utc)
end_dt = end_dt.replace(tzinfo=timezone.utc)

datetime_range = f"{start_dt.isoformat().replace('+00:00','Z')}/{end_dt.isoformat().replace('+00:00','Z')}"
print("STAC datetime:", datetime_range)

# ---- Cache key for manifest ----
manifest_key = sha256_text(
    json.dumps(
        {
            "iso3": ISO3,
            "collection": GFM_COLLECTION_ID,
            "asset_key": GFM_ASSET_KEY,
            "bounds_hash": bounds_hash,
            "window_start": str(pd.to_datetime(start_dt).date()),
            "window_end": str(pd.to_datetime(end_dt).date()),
        },
        sort_keys=True,
    )
)

manifest_json = DIRS["logs"] / f"{ISO3}_stac_manifest_{manifest_key}.json"
manifest_csv = DIRS["logs"] / f"{ISO3}_stac_manifest_{manifest_key}.csv"


def _items_to_df(items_list):
    rows = []
    for it in items_list:
        dt_val = it.datetime.isoformat() if it.datetime is not None else None
        rows.append(
            {
                "id": it.id,
                "datetime": dt_val,
                "collection": it.collection_id,
                "asset_keys": ",".join(sorted(list(it.assets.keys()))),
            }
        )
    return pd.DataFrame(rows).sort_values("datetime")


# ---- Load cached manifest if present ----
if manifest_json.exists() and manifest_csv.exists():
    print("Using cached STAC manifest:", manifest_json.name)
    manifest = json.loads(manifest_json.read_text())
    # Re-fetch items by ids to get fresh hrefs (lightweight, avoids stale signed URLs)
    item_ids = [r["id"] for r in manifest["items"]]
    # STAC API does not guarantee an ID lookup endpoint; easiest is to re-run search.
    # We still keep the cached manifest as provenance, but we refresh the item objects.
    cached_ok = True
else:
    cached_ok = False

# ---- Execute search ----
search = stac_client.search(
    collections=[GFM_COLLECTION_ID],
    datetime=datetime_range,
    intersects=stac_intersects_geom,  # admin2 union polygon
    max_items=None,
)

items = list(search.items())

if not items:
    raise RuntimeError(
        f"No STAC items returned for {ISO3} in {datetime_range} "
        f"for collection={GFM_COLLECTION_ID}"
    )

items_df = _items_to_df(items)

# ---- Persist manifest (provenance) ----
manifest = {
    "iso3": ISO3,
    "collection": GFM_COLLECTION_ID,
    "asset_key": GFM_ASSET_KEY,
    "bounds_hash": bounds_hash,
    "window_start": str(pd.to_datetime(start_dt).date()),
    "window_end": str(pd.to_datetime(end_dt).date()),
    "n_items": int(len(items)),
    "items": items_df.to_dict(orient="records"),
}

manifest_json.write_text(json.dumps(manifest, indent=2))
items_df.to_csv(manifest_csv, index=False)

update_run_metadata_artifact(
    "stac_manifest_json", manifest_json, "STAC search manifest (items list)"
)
update_run_metadata_artifact(
    "stac_manifest_csv", manifest_csv, "STAC search manifest (tabular)"
)
run_metadata["stac_search"] = {
    "datetime_range": datetime_range,
    "n_items": int(len(items)),
    "manifest_key": manifest_key,
}
RUN_METADATA_PATH.write_text(json.dumps(run_metadata, indent=2))

print("Items found:", len(items))
print(
    "First/last item datetime:",
    items_df["datetime"].iloc[0],
    "→",
    items_df["datetime"].iloc[-1],
)
print("Wrote:", manifest_json)
print("Wrote:", manifest_csv)

# Quick check: does expected asset exist?
asset_coverage = items_df["asset_keys"].str.contains(GFM_ASSET_KEY).mean()
print(f"Share of items containing asset '{GFM_ASSET_KEY}': {asset_coverage:.0%}")

STAC datetime: 2024-12-01T00:00:00Z/2025-11-30T00:00:00Z
Using cached STAC manifest: SDN_stac_manifest_fcf089fb90917c764b203a92a8a3b9c766f58b37127ea07e0a805aead1a09bfe.json
Items found: 2608
First/last item datetime: 2024-12-02T04:22:19+00:00 → 2025-11-29T15:45:54+00:00
Wrote: ../outputs/gfm_flood_exposure/SDN/SDN_2024-12-01_2025-11-30_m12/logs/SDN_stac_manifest_fcf089fb90917c764b203a92a8a3b9c766f58b37127ea07e0a805aead1a09bfe.json
Wrote: ../outputs/gfm_flood_exposure/SDN/SDN_2024-12-01_2025-11-30_m12/logs/SDN_stac_manifest_fcf089fb90917c764b203a92a8a3b9c766f58b37127ea07e0a805aead1a09bfe.csv
Share of items containing asset 'ensemble_flood_extent': 100%


In [8]:
# Cell 6 — Load GFM flood extent onto WorldPop grid (cropped) + compute flood_any mask
# Outputs:
#   - geobox_wp: odc.geo.geobox.GeoBox for the WorldPop-aligned country window
#   - flood_ds: xarray.Dataset (lazy, dask-backed) on WorldPop grid/window
#   - flood_da: xarray.DataArray (time, y, x) uint8
#   - flood_any: xarray.DataArray (y, x) bool  (True where flooded at least once in window)

import numpy as np
import rasterio
from rasterio.windows import from_bounds
from odc.stac import load as stac_load
from odc.geo.geobox import GeoBox

# ---- Use manifest_key as the deterministic manifest hash ----
manifest_hash = manifest_key  # stable SHA256 already

# ---- WorldPop grid metadata + window clipped to country bounds ----
with rasterio.open(WORLDPOP_PATH) as wp:
    wp_crs = wp.crs
    wp_transform = wp.transform
    wp_res = wp.res
    wp_bounds = wp.bounds
    wp_width, wp_height = wp.width, wp.height

print("WorldPop grid:")
print("  CRS:", wp_crs)
print("  Shape:", (wp_height, wp_width))
print("  Res:", wp_res)
print("  Bounds:", wp_bounds)

# Country bounds in EPSG:4326 from Cell 3; WorldPop is usually EPSG:4326 in your workflow
minx, miny, maxx, maxy = country_geom_4326.bounds

# Intersect country bounds with WorldPop bounds to ensure window is valid
win_left = max(minx, wp_bounds.left)
win_right = min(maxx, wp_bounds.right)
win_bottom = max(miny, wp_bounds.bottom)
win_top = min(maxy, wp_bounds.top)

if (win_left >= win_right) or (win_bottom >= win_top):
    raise RuntimeError(
        "Country bounds do not intersect WorldPop raster bounds. "
        "Cannot build a WorldPop-aligned window for STAC load."
    )

# Build a rasterio Window for the country extent (clipped to WorldPop coverage)
with rasterio.open(WORLDPOP_PATH) as wp:
    w = from_bounds(win_left, win_bottom, win_right, win_top, transform=wp.transform)
    # Round window to integer pixel grid and clamp to raster
    w = w.round_offsets().round_lengths()
    w = w.intersection(rasterio.windows.Window(0, 0, wp.width, wp.height))

    # Derive an affine for this window
    win_transform = rasterio.windows.transform(w, wp.transform)
    win_width = int(w.width)
    win_height = int(w.height)


# Country bounds (EPSG:4326)
minx, miny, maxx, maxy = country_geom_4326.bounds

# WorldPop bounds
wp_left, wp_bottom, wp_right, wp_top = (
    wp_bounds.left,
    wp_bounds.bottom,
    wp_bounds.right,
    wp_bounds.top,
)

# Intersection bbox = (country ∩ worldpop)
win_left = max(minx, wp_left)
win_right = min(maxx, wp_right)
win_bottom = max(miny, wp_bottom)
win_top = min(maxy, wp_top)

print("Admin2 bounds:", (minx, miny, maxx, maxy))
print("WorldPop bounds:", (wp_left, wp_bottom, wp_right, wp_top))
print("Intersection bbox:", (win_left, win_bottom, win_right, win_top))

# Guard: ensure we actually cropped something
if (
    abs(win_left - wp_left) < 1e-9
    and abs(win_right - wp_right) < 1e-9
    and abs(win_bottom - wp_bottom) < 1e-9
    and abs(win_top - wp_top) < 1e-9
):
    raise RuntimeError(
        "Intersection bbox equals full WorldPop extent; cropping failed. "
        "This would trigger a near-full-raster STAC load."
    )

covers_admin2_extent = (
    wp_left <= minx and wp_bottom <= miny and wp_right >= maxx and wp_top >= maxy
)

extent_gaps = {
    "west": max(0.0, wp_left - minx),
    "south": max(0.0, wp_bottom - miny),
    "east": max(0.0, maxx - wp_right),
    "north": max(0.0, maxy - wp_top),
}

print("WorldPop covers full admin2 extent?", covers_admin2_extent)
print("Extent gaps (deg):", extent_gaps)

run_metadata["worldpop_extent_qc"] = {
    "covers_admin2_extent": bool(covers_admin2_extent),
    "extent_gaps_deg": {k: float(v) for k, v in extent_gaps.items()},
    "admin2_bounds": [float(minx), float(miny), float(maxx), float(maxy)],
    "worldpop_bounds": [
        float(wp_left),
        float(wp_bottom),
        float(wp_right),
        float(wp_top),
    ],
}
RUN_METADATA_PATH.write_text(json.dumps(run_metadata, indent=2, default=str))
from odc.geo.geobox import GeoBox

bbox_win = (win_left, win_bottom, win_right, win_top)  # (left, bottom, right, top)

# WorldPop is square pixels in EPSG:4326, so pass a single float resolution
res = float(wp_res[0])

geobox_wp = GeoBox.from_bbox(
    bbox=bbox_win,
    crs=wp_crs,
    resolution=res,
    tight=True,
)

print("WorldPop-aligned country window:")
print("  shape:", geobox_wp.shape)
print("  crs:", geobox_wp.crs)
print("  res:", geobox_wp.resolution)
print("  extent:", geobox_wp.extent.boundingbox)

# ---- Load GFM flood asset onto the WorldPop window grid ----
# NOTE: This avoids the massive native grid (your earlier ~2TB cube).
# It will still be a big load, but bounded to the country window and at 100m grid.
flood_ds = stac_load(
    items,
    bands=[GFM_ASSET_KEY],
    geobox=geobox_wp,
    chunks={"time": 1, "y": 1024, "x": 1024},
    dtype="uint8",
    fail_on_error=False,  # tolerate occasional corrupt/missing assets
)

if GFM_ASSET_KEY not in flood_ds.data_vars:
    # defensive fallback: take first variable
    v0 = list(flood_ds.data_vars)[0]
    print(f"Warning: expected var '{GFM_ASSET_KEY}' not found; using '{v0}' instead.")
    flood_da = flood_ds[v0]
else:
    flood_da = flood_ds[GFM_ASSET_KEY]

# Nodata handling (from attributes if present)
flood_nodata = flood_da.attrs.get("nodata", None)
if flood_nodata is None:
    # some STAC assets expose nodata in rio accessor or encoding; keep a safe default only if needed
    flood_nodata = 255

print("Flood nodata:", flood_nodata, "dtype:", flood_da.dtype)

# ---- Compute flood-any mask ----
# EODC guidance commonly treats flood presence as values > 0, with nodata=255.
# We exclude nodata explicitly.
flood_any = ((flood_da != flood_nodata) & (flood_da > 0)).any(dim="time")

print("flood_any ready:", flood_any)
print("Dims/sizes:", dict(flood_any.sizes))

# Record to run metadata
run_metadata["gfm_load"] = {
    "manifest_hash": str(manifest_hash),
    "worldpop_window_bounds_epsg4326": {
        "west": float(win_left),
        "south": float(win_bottom),
        "east": float(win_right),
        "north": float(win_top),
    },
    "worldpop_window_shape": [int(win_height), int(win_width)],
    "chunks": {"time": 1, "y": 1024, "x": 1024},
    "flood_nodata": int(flood_nodata) if flood_nodata is not None else None,
}
RUN_METADATA_PATH.write_text(json.dumps(run_metadata, indent=2))

WorldPop grid:
  CRS: EPSG:4326
  Shape: (16252, 20438)
  Res: (0.00083333333, 0.00083333333)
  Bounds: BoundingBox(left=21.814999192739997, bottom=8.681666967939991, right=38.846665791279996, top=22.22500024709999)
Admin2 bounds: (21.81455583500002, 8.637558893000005, 38.58003616299999, 23.146886230000007)
WorldPop bounds: (21.814999192739997, 8.681666967939991, 38.846665791279996, 22.22500024709999)
Intersection bbox: (21.814999192739997, 8.681666967939991, 38.58003616299999, 22.22500024709999)
WorldPop covers full admin2 extent? False
Extent gaps (deg): {'west': 0.0004433577399751698, 'south': 0.04410807493998625, 'east': 0.0, 'north': 0.9218859829000152}
WorldPop-aligned country window:
  shape: Shape2d(x=20119, y=16252)
  crs: GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS

6188

In [9]:
# Cell 7 — Flood mask (any flood in window), nodata-safe + QC + save GeoTIFF
# Patches included:
#   (1) Clip to WorldPop extent BEFORE reducing over time (if WORLDPOP_PATH available)
#   (2) Compute flood_any ONCE and reuse for QC + GeoTIFF write (avoid double scan)
#   (3) Better QC sampling: centre + 4 corners (and report each)

import json
import numpy as np
import xarray as xr
import rioxarray  # noqa: F401
import pandas as pd
import matplotlib.pyplot as plt

FLOOD_NODATA = 255
FLOOD_MASK_NAME = f"{ISO3}_flood_any_{WINDOW_START:%Y-%m-%d}_{WINDOW_END:%Y-%m-%d}"

MASK_DIR = ensure_dir(DIRS["flood_native"] / "flood")
QC_DIR = ensure_dir(DIRS["qc"] / "flood")

# ---- Identify spatial dims robustly ----
dims = flood_da.dims
if "y" in dims and "x" in dims:
    ydim, xdim = "y", "x"
elif "latitude" in dims and "longitude" in dims:
    ydim, xdim = "latitude", "longitude"
elif "lat" in dims and "lon" in dims:
    ydim, xdim = "lat", "lon"
else:
    raise KeyError(f"Unrecognised spatial dims: {dims}")

# Ensure CRS + spatial dims are set for rioxarray operations/writes
if flood_da.rio.crs is None:
    flood_da = flood_da.rio.write_crs("EPSG:4326", inplace=False)

# rioxarray sometimes doesn’t auto-detect x/y dims (esp. lat/lon names)
try:
    _ = flood_da.rio.x_dim
    _ = flood_da.rio.y_dim
except Exception:
    flood_da = flood_da.rio.set_spatial_dims(x_dim=xdim, y_dim=ydim, inplace=False)

# ---- (1) Clip flood raster to WorldPop extent (or intersection if available) ----
# Goal: restrict computation to the same footprint as population calculations.
# Priority order:
#   A) If you already computed INTERSECTION_BBOX = (xmin, ymin, xmax, ymax), use it
#   B) Else if WORLDPOP_PATH exists, read its bounds and clip to those bounds
#   C) Else skip clipping (fallback)

clip_bbox_used = None
try:
    # A) user/previous cell may have already computed this
    if "INTERSECTION_BBOX" in globals() and INTERSECTION_BBOX is not None:
        xmin, ymin, xmax, ymax = INTERSECTION_BBOX
        flood_da = flood_da.rio.clip_box(minx=xmin, miny=ymin, maxx=xmax, maxy=ymax)
        clip_bbox_used = {
            "source": "INTERSECTION_BBOX",
            "bbox": [xmin, ymin, xmax, ymax],
        }
    else:
        # B) derive from WorldPop bounds if WORLDPOP_PATH is defined
        if "WORLDPOP_PATH" in globals() and WORLDPOP_PATH is not None:
            import rasterio

            with rasterio.open(WORLDPOP_PATH) as wp:
                b = wp.bounds  # left, bottom, right, top
            xmin, ymin, xmax, ymax = (
                float(b.left),
                float(b.bottom),
                float(b.right),
                float(b.top),
            )
            flood_da = flood_da.rio.clip_box(minx=xmin, miny=ymin, maxx=xmax, maxy=ymax)
            clip_bbox_used = {
                "source": "WORLDPOP_PATH.bounds",
                "bbox": [xmin, ymin, xmax, ymax],
            }
except Exception as e:
    print("WARNING: Flood->WorldPop clipping skipped due to:", repr(e))
    clip_bbox_used = {"source": "none", "reason": repr(e)}

# ---- Compute nodata-safe any-flood ----
valid = flood_da != FLOOD_NODATA
flooded = (flood_da > 0) & valid
flood_any = flooded.any(dim="time").astype("uint8")  # 1 flooded, 0 not flooded

# ---- (2) Compute ONCE and reuse for QC + write ----
# This avoids scanning the whole grid twice (once for sum(), again for to_raster()).
print("Computing flood_any (single pass) ...")
flood_any_c = flood_any.compute()  # materialise once


# ---- (3) QC sampling: centre + 4 corners on source (time=0) ----
def _sample_window(
    da: xr.DataArray, y0: int, x0: int, h: int = 256, w: int = 256
) -> xr.DataArray:
    return da.isel(
        time=0,
        **{
            ydim: slice(max(0, y0), min(int(da.sizes[ydim]), y0 + h)),
            xdim: slice(max(0, x0), min(int(da.sizes[xdim]), x0 + w)),
        },
    ).compute()


H, W = int(flood_da.sizes[ydim]), int(flood_da.sizes[xdim])
win = 256
anchors = {
    "centre": (H // 2, W // 2),
    "nw": (0, 0),
    "ne": (0, max(0, W - win)),
    "sw": (max(0, H - win), 0),
    "se": (max(0, H - win), max(0, W - win)),
}

qc_samples = {}
for name, (y0, x0) in anchors.items():
    samp = _sample_window(flood_da, y0=y0, x0=x0, h=win, w=win)
    vals, counts = np.unique(samp.values, return_counts=True)
    qc_samples[name] = {int(v): int(c) for v, c in zip(vals, counts)}

print("Flood source sample unique counts by window (time=0):")
for k, v in qc_samples.items():
    print(f"  {k}: {v}")

# Global flooded share (now cheap because flood_any already computed)
flooded_cells = int(flood_any_c.sum().item())
total_cells = int(flood_any_c.size)
flooded_share = (flooded_cells / total_cells) * 100.0 if total_cells else np.nan

print(
    f"Flood_any flooded cells: {flooded_cells:,} / {total_cells:,} ({flooded_share:.3f}%)"
)

# ---- Save the flood_any mask as GeoTIFF ----
out_tif = MASK_DIR / f"{FLOOD_MASK_NAME}.tif"
flood_any_c = flood_any_c.rio.write_nodata(0, inplace=False)  # 0 = not flooded
flood_any_c.rio.to_raster(out_tif, compress="deflate")
print("Wrote flood_any mask:", out_tif)

# ---- QC figure (mask preview + admin2 outlines) ----
fig_path = QC_DIR / f"{FLOOD_MASK_NAME}_preview.png"

# Downsample for plotting speed (does not affect saved raster)
ds_factor = 8
plot_da_ds = flood_any_c.isel(
    **{
        ydim: slice(0, flood_any_c.sizes[ydim], ds_factor),
        xdim: slice(0, flood_any_c.sizes[xdim], ds_factor),
    }
)

fig, ax = plt.subplots(figsize=(10, 8))
plot_da_ds.plot(ax=ax, add_colorbar=False)
admin2_gdf.to_crs("EPSG:4326").boundary.plot(ax=ax, linewidth=0.3, edgecolor="black")
ax.set_title(f"{ISO3} — Any flood extent (window)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
fig.tight_layout()
fig.savefig(fig_path, dpi=200)
plt.close(fig)
print("Wrote QC figure:", fig_path)

# ---- Record metadata ----
run_metadata["flood_mask"] = {
    "mask_tif": str(out_tif),
    "qc_preview_png": str(fig_path),
    "flood_nodata": int(FLOOD_NODATA),
    "clip_bbox_used": clip_bbox_used,
    "source_sample_unique_counts_time0": qc_samples,
    "flooded_cells": flooded_cells,
    "total_cells": total_cells,
    "flooded_share_pct": float(flooded_share) if np.isfinite(flooded_share) else None,
    "dims": list(flood_any_c.dims),
    "shape": [int(flood_any_c.sizes[d]) for d in flood_any_c.dims],
}
RUN_METADATA_PATH.write_text(json.dumps(run_metadata, indent=2, default=str))

Computing flood_any (single pass) ...


/Users/jamesebrown/miniconda3/envs/gfm-stac/lib/python3.11/site-packages/rasterio/warp.py:387: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dest = _reproject(
/Users/jamesebrown/miniconda3/envs/gfm-stac/lib/python3.11/site-packages/rasterio/warp.py:387: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dest = _reproject(
/Users/jamesebrown/miniconda3/envs/gfm-stac/lib/python3.11/site-packages/rasterio/warp.py:387: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dest = _reproject(


Flood source sample unique counts by window (time=0):
  centre: {255: 65536}
  nw: {255: 65536}
  ne: {255: 65536}
  sw: {255: 65536}
  se: {255: 65536}
Flood_any flooded cells: 2,113,211 / 326,973,988 (0.646%)
Wrote flood_any mask: ../outputs/gfm_flood_exposure/SDN/SDN_2024-12-01_2025-11-30_m12/flood_native/flood/SDN_flood_any_2024-12-01_2025-11-30.tif
Wrote QC figure: ../outputs/gfm_flood_exposure/SDN/SDN_2024-12-01_2025-11-30_m12/qc/flood/SDN_flood_any_2024-12-01_2025-11-30_preview.png


6188

In [10]:
# Cell 8 — Population affected raster (WorldPop × flood_any), nodata-safe + QC + GeoTIFF
# Output:
#   - pop_affected_flood_any.tif  (float32; population counts per pixel where flooded, else 0)
#   - QC PNG: WorldPop + pop_affected overlay (downsampled) + admin2 outlines

import json
import numpy as np
import rasterio
import rasterio.warp
import matplotlib.pyplot as plt

# ---- Paths / names ----
POP_NAME = (
    f"{ISO3}_pop_affected_flood_any_{WINDOW_START:%Y-%m-%d}_{WINDOW_END:%Y-%m-%d}"
)
POP_DIR = ensure_dir(DIRS["pop_exposed"] / "flood")  # consistent with other hazards
QC_DIR = ensure_dir(DIRS["qc"] / "flood")

POP_TIF_PATH = POP_DIR / f"{POP_NAME}.tif"
QC_PNG_PATH = QC_DIR / f"{POP_NAME}_preview.png"

# ---- Load WorldPop as reference grid ----
with rasterio.open(WORLDPOP_PATH) as wp:
    wp_arr = wp.read(1).astype("float32")
    wp_transform = wp.transform
    wp_crs = wp.crs
    wp_shape = (wp.height, wp.width)
    wp_nodata = wp.nodata
    wp_bounds = wp.bounds

# Build valid population mask (exclude nodata and negatives)
pop_valid = np.isfinite(wp_arr)
if wp_nodata is not None:
    pop_valid &= wp_arr != float(wp_nodata)
pop_valid &= wp_arr >= 0

# ---- Load flood_any mask GeoTIFF (created in Cell 7) ----
with rasterio.open(out_tif) as src:  # out_tif from Cell 7
    flood_mask_arr = src.read(1).astype("uint8")
    flood_mask_crs = src.crs
    flood_mask_transform = src.transform
    flood_mask_shape = (src.height, src.width)
    flood_mask_bounds = src.bounds

from affine import Affine

# ---- Alignment check (robust, floating-safe) ----
same_shape = flood_mask_shape == wp_shape
same_crs = str(flood_mask_crs) == str(wp_crs)

# Affine transforms should be compared with tolerance, not strict equality
same_transform = (
    isinstance(flood_mask_transform, Affine)
    and isinstance(wp_transform, Affine)
    and flood_mask_transform.almost_equals(wp_transform, precision=1e-9)
)

aligned = same_shape and same_crs and same_transform

if not aligned:
    print("Flood mask not aligned to WorldPop grid:")
    print("  same_shape:", same_shape)
    print("  same_crs:", same_crs)
    print("  same_transform:", same_transform)

# If flood mask doesn’t match WorldPop grid, reproject/resample it to WorldPop grid (nearest)
if not aligned:
    flood_mask_reproj = np.zeros(wp_shape, dtype="uint8")
    rasterio.warp.reproject(
        source=flood_mask_arr,
        destination=flood_mask_reproj,
        src_transform=flood_mask_transform,
        src_crs=flood_mask_crs,
        dst_transform=wp_transform,
        dst_crs=wp_crs,
        resampling=rasterio.warp.Resampling.nearest,
        src_nodata=None,
        dst_nodata=0,
    )
    flood_mask_arr = flood_mask_reproj

# ---- Enforce binary mask semantics (defensive) ----
# Accept any nonzero as flooded; ensure mask is 0/1 afterwards
flood_mask_arr = (flood_mask_arr > 0).astype("uint8")
flooded = flood_mask_arr == 1

# ---- Compute population affected (pixelwise) ----
pop_affected = np.zeros(wp_shape, dtype="float32")
pop_affected[flooded & pop_valid] = wp_arr[flooded & pop_valid]

# ---- QC stats ----
total_pop = float(wp_arr[pop_valid].sum())
affected_pop = float(pop_affected.sum())
affected_pct = (affected_pop / total_pop) * 100.0 if total_pop > 0 else np.nan

affected_cells = int((pop_affected > 0).sum())
valid_cells = int(pop_valid.sum())
flooded_cells = int(flooded.sum())

print("WorldPop nodata:", wp_nodata)
print("WorldPop total pop (valid cells):", f"{total_pop:,.0f}")
print("Flood-affected pop:", f"{affected_pop:,.0f}", f"({affected_pct:.3f}%)")
print("Flood mask aligned to WorldPop grid:", aligned)
print(
    f"Cells: valid_pop={valid_cells:,} flooded_mask={flooded_cells:,} pop_affected_nonzero={affected_cells:,}"
)

# ---- Write GeoTIFF (float32; nodata=0) ----
profile = {
    "driver": "GTiff",
    "height": wp_shape[0],
    "width": wp_shape[1],
    "count": 1,
    "dtype": "float32",
    "crs": wp_crs,
    "transform": wp_transform,
    "nodata": 0.0,
    "compress": "deflate",
    "tiled": True,
    "blockxsize": 512,
    "blockysize": 512,
}

with rasterio.open(POP_TIF_PATH, "w", **profile) as dst:
    dst.write(pop_affected, 1)

print("Wrote pop affected raster:", POP_TIF_PATH)

# ---- QC figure (downsampled) ----
ds = 8  # downsample factor for plotting
wp_plot = np.where(pop_valid, wp_arr, np.nan)[::ds, ::ds]
pa_plot = np.where(pop_affected > 0, pop_affected, np.nan)[::ds, ::ds]

left, bottom, right, top = rasterio.transform.array_bounds(
    wp_shape[0], wp_shape[1], wp_transform
)

fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(wp_plot, extent=(left, right, bottom, top), origin="upper", vmin=0)
ax.imshow(
    pa_plot, extent=(left, right, bottom, top), origin="upper", alpha=0.65, vmin=0
)

admin2_gdf.to_crs("EPSG:4326").boundary.plot(ax=ax, linewidth=0.3, edgecolor="black")
ax.set_title(f"{ISO3} — Flood exposure: WorldPop + affected population (overlay)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
fig.tight_layout()
fig.savefig(QC_PNG_PATH, dpi=200)
plt.close(fig)

print("Wrote QC figure:", QC_PNG_PATH)

# ---- Record metadata ----
run_metadata["flood_pop_affected"] = {
    "pop_tif": str(POP_TIF_PATH),
    "qc_preview_png": str(QC_PNG_PATH),
    "worldpop_path": str(WORLDPOP_PATH),
    "worldpop_nodata": float(wp_nodata) if wp_nodata is not None else None,
    "worldpop_shape": [int(wp_shape[0]), int(wp_shape[1])],
    "worldpop_crs": str(wp_crs),
    "worldpop_bounds": [
        wp_bounds.left,
        wp_bounds.bottom,
        wp_bounds.right,
        wp_bounds.top,
    ],
    "mask_path": str(out_tif),
    "mask_bounds": [
        flood_mask_bounds.left,
        flood_mask_bounds.bottom,
        flood_mask_bounds.right,
        flood_mask_bounds.top,
    ],
    "mask_aligned_to_worldpop": bool(aligned),
    "total_pop_valid_sum": total_pop,
    "pop_affected_sum": affected_pop,
    "pop_affected_pct": float(affected_pct) if np.isfinite(affected_pct) else None,
    "valid_pop_cells": valid_cells,
    "flooded_mask_cells": flooded_cells,
    "pop_affected_nonzero_cells": affected_cells,
}
RUN_METADATA_PATH.write_text(json.dumps(run_metadata, indent=2, default=str))

Flood mask not aligned to WorldPop grid:
  same_shape: False
  same_crs: True
  same_transform: True
WorldPop nodata: -99999.0
WorldPop total pop (valid cells): 50,849,352
Flood-affected pop: 257,257 (0.506%)
Flood mask aligned to WorldPop grid: False
Cells: valid_pop=8,369,760 flooded_mask=2,113,211 pop_affected_nonzero=123,342
Wrote pop affected raster: ../outputs/gfm_flood_exposure/SDN/SDN_2024-12-01_2025-11-30_m12/pop_exposed/flood/SDN_pop_affected_flood_any_2024-12-01_2025-11-30.tif
Wrote QC figure: ../outputs/gfm_flood_exposure/SDN/SDN_2024-12-01_2025-11-30_m12/qc/flood/SDN_pop_affected_flood_any_2024-12-01_2025-11-30_preview.png


6188

In [11]:
# Cell 9 — Flood admin2 aggregation: % population affected
# Output: one CSV row per adm2_pcode with pop_total, pop_affected_flood, pct_affected_flood

import json
import numpy as np
import pandas as pd
import rasterio
from rasterio.features import rasterize
from pathlib import Path
from affine import Affine

# ---- Inputs ----
admin2 = admin2_gdf.copy()
if "adm2_pcode" not in admin2.columns:
    raise KeyError(f"'adm2_pcode' not found. Available columns: {list(admin2.columns)}")

POP_AFFECTED_FLOOD_TIF = (
    Path(DIRS["pop_exposed"])
    / "flood"
    / f"{ISO3}_pop_affected_flood_any_{WINDOW_START:%Y-%m-%d}_{WINDOW_END:%Y-%m-%d}.tif"
)

assert Path(WORLDPOP_PATH).exists(), f"Missing WorldPop: {WORLDPOP_PATH}"
assert (
    POP_AFFECTED_FLOOD_TIF.exists()
), f"Missing pop affected flood raster: {POP_AFFECTED_FLOOD_TIF}"

# ---- Load admin2 minimal columns ----
admin2 = (
    admin2[["adm2_pcode", "geometry"]]
    .dropna(subset=["adm2_pcode", "geometry"])
    .reset_index(drop=True)
)

# ---- Open WorldPop as reference grid ----
with rasterio.open(WORLDPOP_PATH) as wp:
    wp_arr = wp.read(1).astype("float64")
    wp_transform = wp.transform
    wp_crs = wp.crs
    wp_shape = (wp.height, wp.width)
    wp_nodata = wp.nodata

# Robust valid population mask
pop_valid = np.isfinite(wp_arr)
if wp_nodata is not None:
    pop_valid &= wp_arr != float(wp_nodata)
pop_valid &= wp_arr >= 0

# ---- Reproject admin2 to raster CRS and rasterize IDs ----
admin2_wp = admin2.to_crs(wp_crs)

shapes = [(geom, i + 1) for i, geom in enumerate(admin2_wp.geometry)]
admin2_id = rasterize(
    shapes=shapes,
    out_shape=wp_shape,
    transform=wp_transform,
    fill=0,
    dtype="int32",
    all_touched=True,
)

flat_id = admin2_id.ravel()
flat_wp = wp_arr.ravel()
flat_pop_valid = pop_valid.ravel()

valid_cells = (flat_id > 0) & flat_pop_valid
ids_valid = flat_id[valid_cells]
wp_valid_vals = flat_wp[valid_cells]

# Total population per admin2 id
pop_total_by_id = np.bincount(
    ids_valid, weights=wp_valid_vals, minlength=len(admin2_wp) + 1
)

# ---- Load flood pop affected raster and aggregate on same IDs ----
with rasterio.open(POP_AFFECTED_FLOOD_TIF) as src:
    aff_arr = src.read(1).astype("float64")
    aff_nodata = src.nodata

    # Grid alignment checks (tolerant transform comparison)
    same_crs = str(src.crs) == str(wp_crs)
    same_shape = (src.height == wp_shape[0]) and (src.width == wp_shape[1])

    same_transform = (
        isinstance(src.transform, Affine)
        and isinstance(wp_transform, Affine)
        and src.transform.almost_equals(wp_transform, precision=1e-9)
    )

    if not (same_crs and same_shape and same_transform):
        raise ValueError(
            "Flood pop-affected raster is not aligned to WorldPop grid.\n"
            f"WorldPop: crs={wp_crs}, shape={wp_shape}, transform={wp_transform}\n"
            f"Affected: crs={src.crs}, shape={(src.height, src.width)}, transform={src.transform}\n"
            f"Checks: same_crs={same_crs}, same_shape={same_shape}, same_transform={same_transform}"
        )

# Valid mask for affected raster (and require pop_valid so we never count nodata population)
aff_valid = np.isfinite(aff_arr)
if aff_nodata is not None:
    aff_valid &= aff_arr != float(aff_nodata)
aff_valid &= aff_arr >= 0

flat_aff = aff_arr.ravel()
flat_aff_valid = aff_valid.ravel()

valid_aff_cells = valid_cells & flat_aff_valid
ids_valid_aff = flat_id[valid_aff_cells]
aff_valid_vals = flat_aff[valid_aff_cells]

pop_aff_by_id = np.bincount(
    ids_valid_aff, weights=aff_valid_vals, minlength=len(admin2_wp) + 1
)

# ---- Assemble output table ----
n = len(admin2_wp)
admin2_ids = np.arange(1, n + 1, dtype=int)

out = pd.DataFrame(
    {
        "adm2_pcode": admin2_wp["adm2_pcode"].values,
        "pop_total": pop_total_by_id[admin2_ids].astype("float64"),
        "pop_affected_flood": pop_aff_by_id[admin2_ids].astype("float64"),
    }
)

out["pct_affected_flood"] = np.where(
    out["pop_total"] > 0,
    (out["pop_affected_flood"] / out["pop_total"]) * 100.0,
    np.nan,
)

# ---- Save CSV ----
OUT_CSV = (
    Path(DIRS["tables"])
    / f"{ISO3}_admin2_flood_exposure_{WINDOW_START:%Y-%m-%d}_{WINDOW_END:%Y-%m-%d}.csv"
)
out.to_csv(OUT_CSV, index=False)

print("Wrote:", OUT_CSV)
print(out.head(10).to_string(index=False))

# ---- Record metadata ----
run_metadata["admin2_flood_table"] = {
    "path": str(OUT_CSV),
    "n_admin2": int(len(out)),
    "worldpop_path": str(WORLDPOP_PATH),
    "pop_affected_flood_tif": str(POP_AFFECTED_FLOOD_TIF),
    "window_start": str(pd.to_datetime(WINDOW_START).date()),
    "window_end": str(pd.to_datetime(WINDOW_END).date()),
}
RUN_METADATA_PATH.write_text(json.dumps(run_metadata, indent=2, default=str))

Wrote: ../outputs/gfm_flood_exposure/SDN/SDN_2024-12-01_2025-11-30_m12/tables/SDN_admin2_flood_exposure_2024-12-01_2025-11-30.csv
adm2_pcode     pop_total  pop_affected_flood  pct_affected_flood
   SD07090 124069.470488           80.239984            0.064673
   SD16008 138245.265524           16.280536            0.011777
   SD14037 271009.943144         1345.802021            0.496588
   SD05140  90664.168473         1079.882075            1.191079
   SD07088 169557.922108           19.684179            0.011609
   SD05155  66387.326132          598.675283            0.901792
   SD07104  65888.524490           21.312319            0.032346
   SD18028 109610.400664          559.925413            0.510832
   SD18087 189258.042360          505.761335            0.267234
   SD17019 158103.501507           16.608627            0.010505


6188

In [13]:
# Cell 10 — Flood QC pack: summary stats + figures + single PDF export
# Outputs:
#   - PNGs (WorldPop, flood mask, pop affected, admin2 choropleth)
#   - PDF QC pack combining key stats + all figures
#
# Assumes prior cells produced:
#   - out_tif (Cell 7 flood_any mask GeoTIFF)
#   - POP_TIF_PATH + QC_PNG_PATH (Cell 8 pop affected raster)
#   - OUT_CSV (Cell 9 admin2 exposure table)
#   - admin2_gdf (filtered to ISO3), WORLDPOP_PATH, DIRS, ISO3, WINDOW_START, WINDOW_END

import json
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio
from rasterio.features import rasterize
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.patches as mpatches

# ---- Paths ----
QC_DIR = ensure_dir(DIRS["qc"] / "flood")
TABLE_PATH = (
    Path(DIRS["tables"])
    / f"{ISO3}_admin2_flood_exposure_{WINDOW_START:%Y-%m-%d}_{WINDOW_END:%Y-%m-%d}.csv"
)

FLOOD_MASK_TIF = Path(out_tif)  # from Cell 7
POP_AFFECTED_TIF = Path(POP_TIF_PATH)  # from Cell 8

FIG_WP = (
    QC_DIR
    / f"{ISO3}_flood_qc_worldpop_{WINDOW_START:%Y-%m-%d}_{WINDOW_END:%Y-%m-%d}.png"
)
FIG_MASK = (
    QC_DIR / f"{ISO3}_flood_qc_mask_{WINDOW_START:%Y-%m-%d}_{WINDOW_END:%Y-%m-%d}.png"
)
FIG_POP = (
    QC_DIR
    / f"{ISO3}_flood_qc_pop_affected_{WINDOW_START:%Y-%m-%d}_{WINDOW_END:%Y-%m-%d}.png"
)
FIG_CHORO = (
    QC_DIR
    / f"{ISO3}_flood_qc_admin2_pct_{WINDOW_START:%Y-%m-%d}_{WINDOW_END:%Y-%m-%d}.png"
)

PDF_PATH = QC_DIR / f"{ISO3}_flood_QC_{WINDOW_START:%Y-%m-%d}_{WINDOW_END:%Y-%m-%d}.pdf"

assert WORLDPOP_PATH.exists(), f"Missing WorldPop: {WORLDPOP_PATH}"
assert FLOOD_MASK_TIF.exists(), f"Missing flood_any mask: {FLOOD_MASK_TIF}"
assert POP_AFFECTED_TIF.exists(), f"Missing pop affected: {POP_AFFECTED_TIF}"
assert TABLE_PATH.exists(), f"Missing admin2 table: {TABLE_PATH}"

# ---- Load admin2 + table ----
admin2 = admin2_gdf.copy()
if "adm2_pcode" not in admin2.columns:
    raise KeyError(f"'adm2_pcode' not found. Columns: {list(admin2.columns)}")

tbl = pd.read_csv(TABLE_PATH)
if "adm2_pcode" not in tbl.columns:
    raise KeyError(f"'adm2_pcode' missing from table. Columns: {list(tbl.columns)}")

# Join table back to geoms for choropleth
admin2_tbl = admin2.merge(tbl, on="adm2_pcode", how="left")

# ---- Load rasters ----
with rasterio.open(WORLDPOP_PATH) as wp:
    wp_arr = wp.read(1).astype("float32")
    wp_transform = wp.transform
    wp_crs = wp.crs
    wp_shape = (wp.height, wp.width)
    wp_nodata = wp.nodata
    wp_bounds = wp.bounds

# Valid mask for plotting/stats
wp_valid = np.isfinite(wp_arr)
if wp_nodata is not None:
    wp_valid &= wp_arr != float(wp_nodata)
wp_valid &= wp_arr >= 0

with rasterio.open(FLOOD_MASK_TIF) as fm:
    flood_mask = fm.read(1).astype("uint8")
    fm_nodata = fm.nodata
    fm_crs = fm.crs
    fm_transform = fm.transform

with rasterio.open(POP_AFFECTED_TIF) as pa:
    pop_aff = pa.read(1).astype("float32")
    pa_nodata = pa.nodata
    pa_crs = pa.crs
    pa_transform = pa.transform

# ---- Basic sanity checks (should already align by design) ----
if (str(fm_crs) != str(wp_crs)) or (flood_mask.shape != wp_shape):
    print(("Flood mask raster not on WorldPop grid (CRS/shape mismatch)."))
    # raise ValueError("Flood mask raster not on WorldPop grid (CRS/shape mismatch).")

if (str(pa_crs) != str(wp_crs)) or (pop_aff.shape != wp_shape):
    print(("Pop-affected raster not on WorldPop grid (CRS/shape mismatch)."))
    # raise ValueError("Pop-affected raster not on WorldPop grid (CRS/shape mismatch).")

# ---- QC stats ----
total_pop = float(wp_arr[wp_valid].sum())
affected_pop = float(np.where(np.isfinite(pop_aff) & (pop_aff > 0), pop_aff, 0.0).sum())
affected_pct = (affected_pop / total_pop) * 100.0 if total_pop > 0 else np.nan

flooded_cells = int((flood_mask == 1).sum())
total_cells = int(flood_mask.size)
flooded_share = (flooded_cells / total_cells) * 100.0 if total_cells else np.nan

top10 = tbl.sort_values("pct_affected_flood", ascending=False).head(10)[
    ["adm2_pcode", "pop_total", "pop_affected_flood", "pct_affected_flood"]
]

print("QC summary:")
print(f"  WorldPop valid total pop: {total_pop:,.0f}")
print(f"  Flood-affected pop:       {affected_pop:,.0f} ({affected_pct:.3f}%)")
print(
    f"  Flooded cells (mask=1):   {flooded_cells:,} / {total_cells:,} ({flooded_share:.3f}%)"
)
print("  Top 10 admin2 by % affected:")
print(top10.to_string(index=False))


# ---- Plot helpers ----
def raster_extent(bounds):
    # bounds: left, bottom, right, top
    return (bounds.left, bounds.right, bounds.bottom, bounds.top)


extent = raster_extent(wp_bounds)

# Keep admin2 outlines thin black lines on all figures
admin2_outline = admin2.to_crs(wp_crs).boundary

# Downsample for PNG speed (does not affect data outputs)
DS = 8
wp_plot = np.where(wp_valid, wp_arr, np.nan)[::DS, ::DS]
mask_plot = np.where(flood_mask == 1, 1.0, np.nan)[::DS, ::DS]
pa_plot = np.where(pop_aff > 0, pop_aff, np.nan)[::DS, ::DS]

# ---- Figure 1: WorldPop (nodata-safe) ----
fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(wp_plot, extent=extent, origin="upper")
admin2_outline.plot(ax=ax, linewidth=0.3, edgecolor="black")
ax.set_title(f"{ISO3} — WorldPop (valid cells only)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
fig.tight_layout()
fig.savefig(FIG_WP, dpi=200)
plt.close(fig)

# ---- Figure 2: Flood mask ----
fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(mask_plot, extent=extent, origin="upper")
admin2_outline.plot(ax=ax, linewidth=0.3, edgecolor="black")
ax.set_title(f"{ISO3} — Flood mask (any flood in window)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
fig.tight_layout()
fig.savefig(FIG_MASK, dpi=200)
plt.close(fig)

# ---- Figure 3: Population affected (overlay on WorldPop) ----
fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(wp_plot, extent=extent, origin="upper")
ax.imshow(pa_plot, extent=extent, origin="upper", alpha=0.65)
admin2_outline.plot(ax=ax, linewidth=0.3, edgecolor="black")
ax.set_title(f"{ISO3} — Population affected by flood (overlay on WorldPop)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
fig.tight_layout()
fig.savefig(FIG_POP, dpi=200)
plt.close(fig)

# ---- Figure 4: Admin2 choropleth (% affected) ----
# Plot in EPSG:4326 for consistency; admin2_tbl already aligned by merge
admin2_tbl_4326 = admin2_tbl.to_crs("EPSG:4326")

fig, ax = plt.subplots(figsize=(10, 8))
admin2_tbl_4326.plot(
    column="pct_affected_flood",
    ax=ax,
    legend=True,
    missing_kwds={"color": "lightgrey", "label": "Missing"},
)
admin2_tbl_4326.boundary.plot(ax=ax, linewidth=0.3, edgecolor="black")
ax.set_title(f"{ISO3} — Admin2 % population affected by flood")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
fig.tight_layout()
fig.savefig(FIG_CHORO, dpi=200)
plt.close(fig)

print("Wrote QC PNGs:")
print(" ", FIG_WP)
print(" ", FIG_MASK)
print(" ", FIG_POP)
print(" ", FIG_CHORO)

# ---- Build PDF QC pack ----
with PdfPages(PDF_PATH) as pdf:
    # Page 1: text summary
    fig, ax = plt.subplots(figsize=(8.27, 11.69))  # A4 portrait in inches-ish
    ax.axis("off")

    lines = []
    lines.append(f"Flood QC Pack — {ISO3}")
    lines.append(f"Window: {WINDOW_START:%Y-%m-%d} → {WINDOW_END:%Y-%m-%d}")
    lines.append("")
    lines.append("Key inputs")
    lines.append(f"  WorldPop: {WORLDPOP_PATH}")
    lines.append(f"  Flood mask: {FLOOD_MASK_TIF}")
    lines.append(f"  Pop affected: {POP_AFFECTED_TIF}")
    lines.append(f"  Admin2 table: {TABLE_PATH}")
    lines.append("")
    lines.append("QC summary")
    lines.append(f"  WorldPop nodata: {wp_nodata}")
    lines.append(f"  WorldPop valid total pop: {total_pop:,.0f}")
    lines.append(
        f"  Flooded cells (mask=1): {flooded_cells:,} / {total_cells:,} ({flooded_share:.3f}%)"
    )
    lines.append(f"  Flood-affected pop: {affected_pop:,.0f} ({affected_pct:.3f}%)")
    lines.append("")
    lines.append("Top 10 admin2 by % affected")
    for _, r in top10.iterrows():
        lines.append(
            f"  {r['adm2_pcode']}: {r['pct_affected_flood']:.2f}% "
            f"({r['pop_affected_flood']:,.0f} / {r['pop_total']:,.0f})"
        )

    ax.text(
        0.02,
        0.98,
        "\n".join(lines),
        va="top",
        ha="left",
        fontsize=10,
        family="monospace",
    )
    fig.tight_layout()
    pdf.savefig(fig)
    plt.close(fig)

    # Pages: figures
    for p in [FIG_WP, FIG_MASK, FIG_POP, FIG_CHORO]:
        img = plt.imread(p)
        fig, ax = plt.subplots(figsize=(11.69, 8.27))  # A4 landscape
        ax.imshow(img)
        ax.axis("off")
        fig.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

print("Wrote QC PDF:", PDF_PATH)

# ---- Record to metadata ----
run_metadata["flood_qc_pack"] = {
    "qc_pngs": [str(FIG_WP), str(FIG_MASK), str(FIG_POP), str(FIG_CHORO)],
    "qc_pdf": str(PDF_PATH),
    "summary": {
        "worldpop_total_pop_valid_sum": total_pop,
        "flooded_cells": flooded_cells,
        "mask_total_cells": total_cells,
        "flooded_share_pct": (
            float(flooded_share) if np.isfinite(flooded_share) else None
        ),
        "pop_affected_sum": affected_pop,
        "pop_affected_pct": float(affected_pct) if np.isfinite(affected_pct) else None,
    },
}
RUN_METADATA_PATH.write_text(json.dumps(run_metadata, indent=2, default=str))

Flood mask raster not on WorldPop grid (CRS/shape mismatch).
QC summary:
  WorldPop valid total pop: 50,849,352
  Flood-affected pop:       257,257 (0.506%)
  Flooded cells (mask=1):   2,113,211 / 326,973,988 (0.646%)
  Top 10 admin2 by % affected:
adm2_pcode     pop_total  pop_affected_flood  pct_affected_flood
   SD03172  50396.456157        14810.911393           29.388795
   SD03159  62940.503982        11105.855476           17.645006
   SD03146  64989.354861        10949.614188           16.848320
   SD03158 181855.153444        27523.955008           15.135098
   SD03141  73058.223516         6388.848872            8.744873
   SD03143 363571.505131        28850.408397            7.935278
   SD06110  75432.313196         4818.358190            6.387658
   SD03157 223894.747193        13640.470069            6.092358
   SD03154  33731.678993         1394.290808            4.133476
   SD03166 190072.076463         7793.186902            4.100122
Wrote QC PNGs:
  ../outputs/gfm_floo

7162